# Beginner Tutorial 4: Caching — Avoiding Redundant Work

## What You Will Learn

- What caching is and why it matters
- How hash functions create "fingerprints" of data
- What content-addressable storage means
- How to use the `@cacheable` decorator
- How to handle file-based inputs
- Cache invalidation strategies

## Prerequisites

- Completed notebook 01 (Getting Started)
- `pip install scalable`

In [ ]:
import os
import tempfile
import time

project_dir = tempfile.mkdtemp(prefix="scalable-beginner-04-")
os.chdir(project_dir)
os.makedirs("cache", exist_ok=True)
print(f"Working directory: {project_dir}")

## 💡 Key Concept: What is Caching?

**Caching** = storing results so you don't recompute them.
It trades **storage space** for **computation time**.

Real-world examples:
- Browser cache: stores images so pages load faster on revisit
- CPU cache: keeps frequently-used data close to processor

In Scalable: "If I've already computed `f(x)` and saved the result,
don't compute it again — just return the saved result."

## 💡 Key Concept: Hash Functions

A **hash function** takes any input and produces a fixed-size "fingerprint":

```
"Hello"      → a3b8c9d2...  (always same for same input)
"Hello!"     → 7f83b165...  (tiny change → totally different hash)
(500MB file) → e1f0a2b3...  (same fixed size regardless of input)
```

Key properties: deterministic, fixed-size output, one-way.

In [ ]:
# Demonstrate hashing
import hashlib

# Same input → same hash (deterministic)
hash1 = hashlib.sha256(b"Hello, World!").hexdigest()[:16]
hash2 = hashlib.sha256(b"Hello, World!").hexdigest()[:16]
print(f"Hash of 'Hello, World!': {hash1}")
print(f"Hash again (same):      {hash2}")
print(f"Same? {hash1 == hash2}")

# Tiny change → completely different hash
hash3 = hashlib.sha256(b"Hello, World!!").hexdigest()[:16]
print(f"\nHash of 'Hello, World!!': {hash3}")
print(f"Completely different! This is the 'avalanche effect'.")

## 💡 Key Concept: Python Decorators

A **decorator** wraps a function to add behavior without changing its code:

```python
@some_decorator      # ← This wraps the function below
def my_function(x):
    return x * 2
```

Scalable's `@cacheable` decorator adds: "check cache before computing, save result after computing."

In [ ]:
# Set up environment for caching
os.environ['SCALABLE_CACHE_DIR'] = os.path.join(project_dir, 'cache')

from scalable import cacheable

# Explanation: @cacheable tells Scalable to cache this function's results
# return_type=dict: tells Scalable how to serialize the result
# scenario_id=int: tells Scalable how to hash this argument
@cacheable(return_type=dict, scenario_id=int)
def expensive_simulation(scenario_id: int) -> dict:
    """Simulate expensive computation (2 seconds)."""
    time.sleep(2)  # Pretend this takes 2 seconds
    return {
        "scenario_id": scenario_id,
        "result": scenario_id * 42,
        "computed": True,
    }

print("Function decorated with @cacheable")
print("First call will be slow (cache miss), second will be instant (cache hit)")

In [ ]:
# FIRST CALL: Cache miss (slow — has to compute)
start = time.time()
result1 = expensive_simulation(scenario_id=42)
first_call_time = time.time() - start

print(f"First call (cache MISS): {first_call_time:.2f}s")
print(f"Result: {result1}")

In [ ]:
# SECOND CALL: Cache hit (instant — returns saved result)
start = time.time()
result2 = expensive_simulation(scenario_id=42)
second_call_time = time.time() - start

print(f"Second call (cache HIT): {second_call_time:.4f}s")
print(f"Result: {result2}")
print(f"\n💡 Speedup: {first_call_time / max(second_call_time, 0.001):.0f}x faster!")
print(f"Same result? {result1 == result2}")

In [ ]:
# DIFFERENT INPUT: Cache miss (different scenario_id = different cache key)
start = time.time()
result3 = expensive_simulation(scenario_id=99)
third_call_time = time.time() - start

print(f"Different input (cache MISS): {third_call_time:.2f}s")
print(f"Result: {result3}")
print(f"\n💡 Different input → different cache key → must recompute")

## 🤔 Think About It: Cache Invalidation

What if you **fix a bug** in your function but the inputs stay the same?

The cache key hasn't changed (same function name + same arguments),
so you'll get the OLD (buggy) result from cache!

**Solutions:**
1. Clear the cache: `rm -rf ./cache/`
2. Version your function: add `_version="2"` to the decorator

This is why cache invalidation is one of the "two hard things in computer science."

## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Caching | Storing results for reuse (trade storage for time) |
| Hash Function | Fixed-size fingerprint of any input |
| Content-Addressable Storage | Data addressed by content hash, not name |
| Memoization | Caching function results based on inputs |
| Decorator | Python pattern wrapping a function to add behavior |
| Cache Key | Unique identifier (hash of function + args) |
| Cache Hit | Result found in cache (fast!) |
| Cache Miss | Result not found, must compute |
| Cache Invalidation | Deciding when cached results are stale |
| Serialization | Converting objects to bytes for storage |
| Determinism | Same inputs always produce same output |

In [ ]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")